In [15]:
!nvidia-smi

Mon Aug 31 04:19:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
!ls -la /content/drive/MyDrive/silent_speech

total 10533191
-rw------- 1 root root 6866471136 Aug 24 03:45 emg_dataset.h5
-rw------- 1 root root 3919507637 Aug 24 03:45 emg_data.tar.gz
drwx------ 2 root root       4096 Aug 24 03:49 KenLM
drwx------ 2 root root       4096 Aug 31 04:18 output


In [18]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!git submodule update --init text_alignments
!tar -xzf text_alignments/text_alignments.tar.gz
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
fatal: destination path 'silent_speech' already exists and is not an empty directory.
/content/silent_speech
fatal: not a git repository (or any of the parent directories): .git
tar (child): text_alignments/text_alignments.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
sed: can't read architecture.py: No such file or directory


In [19]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy

In [20]:
%env DATA_PATH=/content/data

env: DATA_PATH=/content/data


In [21]:
%cd /content/silent_speech
!mkdir -p /content/data/Gaddy/h5
!cp /content/drive/MyDrive/silent_speech/emg_data.tar.gz /content/data/Gaddy/
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/


/content/silent_speech


In [22]:
!python download_data.py

python3: can't open file '/content/silent_speech/download_data.py': [Errno 2] No such file or directory


In [23]:
# CONTINUE training
import glob, os, torch
out = "/content/drive/MyDrive/silent_speech/output"
ckpts = sorted(glob.glob(os.path.join(out, "model_*_last.pt")), key=os.path.getmtime)
assert ckpts, "no _last.pt found on Drive"
print("resuming from:", ckpts[-1])
sd = torch.load(ckpts[-1], map_location="cpu", weights_only=False)
torch.save({"state_dict": sd}, os.path.join(out, "resume.pt"))
print("wrote resume.pt")

resuming from: /content/drive/MyDrive/silent_speech/output/model_20260824_061030_last.pt
wrote resume.pt


In [24]:
import os, json
p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg["start_training_from"] = "/content/drive/MyDrive/silent_speech/output/resume.pt"
cfg["ckpt_directory"]      = "/content/drive/MyDrive/silent_speech/output"
cfg["num_epochs"]          = 200
cfg["eval_interval"]       = 5
cfg["num_workers"]         = os.cpu_count()   # was 2 — use all cores (~12)
json.dump(cfg, open(p, "w"), indent=4)
print("num_workers =", cfg["num_workers"], "| CPUs available:", os.cpu_count())

FileNotFoundError: [Errno 2] No such file or directory: '/content/silent_speech/config/recognition_model.json'

In [ ]:
import os, glob
os.chdir("/content/silent_speech")
os.environ["BEST"] = sorted(
    glob.glob("/content/drive/MyDrive/silent_speech/**/model_*_best.pt", recursive=True),
    key=os.path.getmtime)[-1]
print("evaluating:", os.environ["BEST"])
!python recognition_model.py --evaluate_saved "$BEST"